# SmartCampus AI — comparaison des LLM gratuits sur GPU Kaggle / Colab (SA-31)

Compare **tous les modèles open-source candidats** (Qwen, Llama, Mistral, Aya, Gemma, Phi, Command R7B Arabic…) avec
le **même** `evaluate.py` et le **même** `dataset.jsonl` (66 questions FR / tunisien / Arabizi) que sur le PC.
Aucune API payante : les modèles tournent dans le notebook, sur le GPU gratuit.

**Pour chaque modèle** : téléchargement → évaluation → résultats enregistrés → **suppression du modèle** (le disque
ne se remplit pas). Si la session coupe, relancez les cellules : les modèles déjà évalués sont sautés.

## Avant de commencer
1. **Kaggle** (recommandé, ~30 h de GPU/semaine) : *Settings → Accelerator → GPU T4 x2* et *Internet → On*
   (Kaggle demande de vérifier le compte par téléphone pour activer GPU + Internet).
   **Colab** : *Exécution → Modifier le type d'exécution → GPU T4*.
2. Importer `smartcampus-eval.zip` (contenu : `llm-eval/` + `backend/app/`, commande en bas du notebook) :
   Kaggle → *Add Input → Upload* ; Colab → panneau *Fichiers*.
3. *Run all*. Durée indicative : 5 à 15 min par modèle 7–9B sur T4.

In [ ]:
# 1) Installer Ollama et les dépendances Python du banc
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install httpx pydantic-settings

In [ ]:
# 2) Démarrer le serveur Ollama sur le GPU
import os, subprocess, time, httpx
os.environ["OLLAMA_KEEP_ALIVE"] = "10m"
server = subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        httpx.get("http://localhost:11434/api/tags", timeout=2); print("Ollama prêt"); break
    except Exception:
        time.sleep(1)
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().replace("\n", " + ")
print("GPU :", GPU or "AUCUN — activez l'accélérateur GPU !")

In [ ]:
# 3) Récupérer le code du banc
import glob
zips = glob.glob("/kaggle/input/**/smartcampus-eval.zip", recursive=True) + glob.glob("/content/smartcampus-eval.zip")
if zips:
    !unzip -oq "{zips[0]}" -d /tmp/smartcampus
else:
    # une fois le dépôt poussé sur GitHub (public), on peut cloner au lieu d'importer le zip
    !git clone -q --depth 1 -b develop https://github.com/MohamedTrikii/SmartCampus_IAGenProject /tmp/smartcampus
os.chdir("/tmp/smartcampus/llm-eval")
!wc -l dataset.jsonl
# résultats dans un dossier persistant et téléchargeable
OUT = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "/content/results"
os.makedirs(OUT, exist_ok=True); print("Résultats :", OUT)

## 4) Catalogue des modèles

Tags de la bibliothèque Ollama (quantification Q4 par défaut). Un T4 a 16 Go : jusqu'à ~12–14B. Les tags
indisponibles sont simplement signalés et sautés. Commentez une ligne pour ne pas tester un modèle.

**Pourquoi pas GPT-2 ?** GPT-2 (2019) n'est pas un modèle conversationnel : il ne suit pas d'instructions, ne sait
pas répondre en JSON, a été entraîné presque uniquement en anglais et n'a que 1 024 tokens de contexte (notre prompt
système + catalogue n'y tient pas). Il obtiendrait ~0 partout : inutile de consommer du GPU pour le vérifier.
Même remarque pour les modèles « base » non instruits et les modèles de raisonnement (`deepseek-r1`) dont les
balises `<think>` cassent la sortie JSON et multiplient la latence.

In [ ]:
MODELS = [
    # --- petits (≤ 4B) : utilisables aussi sur un PC sans GPU puissant
    "qwen2.5:3b",
    "llama3.2:3b",
    "gemma2:2b",
    "phi3.5:3.8b",
    "qwen3:4b",
    "gemma3:4b",
    # --- moyens (7–9B) : candidats principaux pour le serveur de démo et le fine-tuning LoRA
    "qwen2.5:7b-instruct",
    "qwen3:8b",
    "llama3.1:8b",
    "mistral:7b",
    "aya-expanse:8b",            # Cohere, conçu pour le multilingue dont l'arabe
    "command-r7b-arabic:7b",     # Cohere, spécialisé arabe
    "gemma2:9b",
    # --- grands (12B) : tiennent encore sur un T4
    "mistral-nemo:12b",
    "gemma3:12b",
]
print(len(MODELS), "modèles à évaluer")

In [ ]:
# 5) Boucle : télécharger → évaluer → supprimer (reprise automatique si la session a coupé)
import json, re, shutil
status = {}
for model in MODELS:
    done = os.path.join(OUT, re.sub(r"[^\w.-]", "_", model) + "_summary.json")
    if os.path.exists(done):
        status[model] = "déjà évalué"; print(f"✓ {model} déjà évalué"); continue
    print(f"\n######## {model} ########")
    pull = subprocess.run(["ollama", "pull", model], capture_output=True, text=True)
    if pull.returncode != 0:
        status[model] = "indisponible : " + (pull.stderr.strip().splitlines() or ["?"])[-1]
        print("✗", status[model]); continue
    t0 = time.time()
    run = subprocess.run(["python", "evaluate.py", "--models", model, "--skip-done", "--out-dir", OUT,
                          "--machine", GPU, "--base-url", "http://localhost:11434/v1"],
                         capture_output=True, text=True, env={**os.environ, "LLM_PROVIDER": "openai"})
    print(run.stdout[-1500:], run.stderr[-800:])
    status[model] = f"évalué en {(time.time() - t0) / 60:.1f} min" if os.path.exists(done) else "échec de l'évaluation"
    subprocess.run(["ollama", "rm", model], capture_output=True)  # libère le disque avant le modèle suivant
    print("→", status[model], "| disque libre :", shutil.disk_usage("/").free // 2**30, "Go")
print("\n", json.dumps(status, ensure_ascii=False, indent=1))

In [ ]:
# 6) Tableau comparatif (trié par score) + archive à télécharger
!python evaluate.py --summary-only --out-dir "$OUT" > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(OUT, "summary.md"), encoding="utf-8").read()))
shutil.make_archive(OUT, "zip", OUT); print("À télécharger :", OUT + ".zip")

## Préparer l'archive sur votre PC

Depuis la racine du dépôt — PowerShell :

```powershell
Compress-Archive -Path llm-eval, backend\app -DestinationPath smartcampus-eval.zip -Force
```

Git Bash : `zip -r smartcampus-eval.zip llm-eval backend/app -x '*/__pycache__/*' 'llm-eval/results/*'`

## Après l'exécution

1. Télécharger `results.zip` et le décompresser dans `llm-eval/results/` sur le PC.
2. Notation humaine : chaque membre remplit les colonnes des fichiers `*_notation_humaine.csv` (Excel, séparateur `;`).
3. Remplir `docs/choix-llm.md` **uniquement avec ces chiffres**, en indiquant le GPU (colonne `machine`).
4. Les latences ne se comparent qu'entre modèles évalués sur la même machine.